In [ ]:
import pandas as pd

ROOT = r'/Users/Zac/PycharmProjects/Cartcounting_model_forecast'

# Decisions published by the ML server (comma-separated export of the
# receiver's scoring_results table).
data_alert = pd.read_csv(r'/Users/Zac/PycharmProjects/Cartcounting_model_forecast/cart_platform/outputs/experiment_1.csv')

data_alert['decision_time'] = pd.to_datetime(data_alert['decision_time'])

# Raw telemetry is tab-separated with a BOM.
data_raw = pd.read_csv(r'/Users/Zac/Downloads/cart_count_163_215_0_2ebecfe8-1d8d-4c91-88db-f3a25de710b8.csv')
# data_raw = pd.read_csv(r'/Users/Zac/Downloads/cart_count_20261026_095221_014.csv')
data_raw['DVRDate'] = pd.to_datetime(data_raw['DVRDate'])
data_raw['next_DVRDate'] = pd.to_datetime(data_raw['next_DVRDate'])
print(data_alert.shape, data_raw.shape)

In [ ]:
data_alert['mmdd'] = data_alert['decision_time'].dt.strftime('%m-%d')
data_alert[data_alert['mmdd']=='10-22']

In [ ]:
data_alert

In [ ]:
final_result = data_raw.merge(

    data_alert[['decision_time','alert_state','shortage_risk','quantity','urgency',  'new_predictive_alert', 'reactive_critical', 'expected_completion_time',
       'actionable_before_breach', 'minutes_to_predicted_breach','reason']],
# should_replenish
    how='left',

    left_on='next_DVRDate',

    right_on='decision_time'

)
final_result = final_result[final_result['alert_state'].notnull()]

In [ ]:
final_result['mmdd'] = final_result['decision_time'].dt.strftime('%m-%d')


In [ ]:
final_result.groupby('mmdd')['alert_state'].value_counts().unstack()

In [ ]:
final_result

In [ ]:
final_result[
    (final_result["alert_state"] == "WARNING") &
    (final_result["quantity"] > 0)
].shape,final_result[
    final_result["alert_state"] == "CRITICAL"
].shape

In [ ]:
 # python3 scripts/run_experiment.py \
 #  --base-url http://127.0.0.1:8100 \
 #  --entity 168:194:13 --bay-id 540f7e8b_bf61_47fa_a812_a8fa45c7ad56 \
 #  --data /Users/Zac/Downloads/cart_count_168_194_13_540F7E8B-BF61-47FA-A812-A8FA45C7AD56.csv \
 #  --out outputs/experiment.csv

In [ ]:
df = pd.read_csv(r'/Users/Zac/Downloads/cart_count_163_215_0_2ebecfe8-1d8d-4c91-88db-f3a25de710b8.csv')

df = df[df['next_DVRDate']!= ' '].loc[:, ~df.columns.str.startswith("Unnamed:")]
df.to_csv(
    "/Users/Zac/Downloads/cart_count_163_215_0_2ebecfe8-1d8d-4c91-88db-f3a25de710b8.csv",
    index=False,
)
df

# 151:251:0:cef95f3e_6810_4216_829c_eb8143576bcc

In [ ]:
final_result

In [ ]:
final_result = pd.read_csv(r'/Users/Zac/PycharmProjects/Cartcounting_model_forecast/cart_platform/outputs/cms_phuoc_dbo_scoring_results.csv')
final_result['decision_time'] = pd.to_datetime(final_result['decision_time'])
final_result = final_result[final_result['entity_id']=='163:215:0:2ebecfe8_1d8d_4c91_88db_f3a25de710b8']
final_result

In [ ]:
final_result['entity_id'].unique()
# [ '151:251:0:cef95f3e_6810_4216_829c_eb8143576bcc',
#   '163:215:0:2ebecfe8_1d8d_4c91_88db_f3a25de710b8',
#  '168:194:13:540f7e8b_bf61_47fa_a812_a8fa45c7ad56',
#   '188:165:4:4870e2bd_f9ab_49b3_b57d_4e6344e79d1b']

In [18]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import html
import textwrap


# ============================================================
# 1. CONFIGURATION
# ============================================================

AMBER_THRESHOLD = 17
RED_THRESHOLD = 2

# WARNING quantity > 0
# CRITICAL: all
NOTIFICATION_MIN_QUANTITY = 0

# Resolve after 3 consecutive recovery records
# NO time-gap validation
RECOVERY_REQUIRED = 3

OUTPUT_FILE = "cart_alert_resolution_dashboard.html"

URGENCY_COLORS = {
    "LOW": "#22C55E",
    "MEDIUM": "#F59E0B",
    "HIGH": "#F97316",
    "URGENT": "#DC2626"
}

EVENT_STYLES = {
    "Predictive Alert": {
        "color": "#2563EB",
        "symbol": "star",
        "size": 15
    },
    "Reactive Critical": {
        "color": "#991B1B",
        "symbol": "x",
        "size": 15
    },
    "Replenishment Order": {
        "color": "#F97316",
        "symbol": "diamond",
        "size": 12
    },
    "Expected Completion": {
        "color": "#7C3AED",
        "symbol": "square",
        "size": 11
    },
    "Predicted Breach": {
        "color": "#DC2626",
        "symbol": "triangle-down",
        "size": 12
    },
    "Resolved Alert": {
        "color": "#16A34A",
        "symbol": "circle",
        "size": 16
    }
}

PLOTLY_CONFIG = {
    "displayModeBar": True,
    "scrollZoom": True,
    "displaylogo": False,
    "responsive": True
}


# ============================================================
# 2. LOAD DATA
# ============================================================



if "final_result" in globals():

    df = final_result.copy()

else:

    df = pd.read_csv("Out_9.csv")


# ============================================================
# 3. DATA PREPROCESSING
# ============================================================

# ------------------------------------------------------------
# Cart Count
# ------------------------------------------------------------

if "Count" not in df.columns:

    if "current_cart" in df.columns:

        df["Count"] = df["current_cart"]

    else:

        raise ValueError(
            "Dataset must contain Count or current_cart."
        )


# ------------------------------------------------------------
# Entity
# ------------------------------------------------------------

if "entity_id" not in df.columns:

    df["entity_id"] = "default_entity"

df["entity_id"] = (
    df["entity_id"]
    .fillna("default_entity")
    .astype(str)
)


# ------------------------------------------------------------
# Datetime
# ------------------------------------------------------------

def parse_datetime(series):

    try:

        return pd.to_datetime(
            series,
            errors="coerce",
            format="mixed"
        )

    except (TypeError, ValueError):

        return pd.to_datetime(
            series,
            errors="coerce"
        )


if "decision_time" not in df.columns:

    raise ValueError(
        "Dataset must contain decision_time."
    )

df["decision_time"] = parse_datetime(
    df["decision_time"]
)

if "expected_completion_time" not in df.columns:

    df["expected_completion_time"] = pd.NaT

df["expected_completion_time"] = parse_datetime(
    df["expected_completion_time"]
)


# ------------------------------------------------------------
# Numeric columns
# ------------------------------------------------------------

numeric_cols = [
    "Count",
    "shortage_risk",
    "quantity",
    "should_replenish",
    "minutes_to_predicted_breach"
]

for col in numeric_cols:

    if col not in df.columns:

        df[col] = np.nan

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

df["quantity"] = df["quantity"].fillna(0)


# ------------------------------------------------------------
# Alert State
# ------------------------------------------------------------

if "alert_state" not in df.columns:

    df["alert_state"] = "NORMAL"

df["alert_state"] = (
    df["alert_state"]
    .fillna("NORMAL")
    .astype(str)
    .str.upper()
    .str.strip()
)


# ------------------------------------------------------------
# Urgency
# ------------------------------------------------------------

if "urgency" not in df.columns:

    df["urgency"] = "NONE"

df["urgency"] = (
    df["urgency"]
    .fillna("NONE")
    .astype(str)
    .str.upper()
    .str.strip()
)


# ------------------------------------------------------------
# Reason
# ------------------------------------------------------------

if "reason" not in df.columns:

    df["reason"] = pd.NA


def format_reason(value, width=65):

    if pd.isna(value):

        return "N/A"

    value = str(value).strip()

    if not value or value.lower() in [
        "nan",
        "none",
        "null"
    ]:

        return "N/A"

    value = html.escape(value)

    lines = textwrap.wrap(
        value,
        width=width,
        break_long_words=False,
        break_on_hyphens=False
    )

    return "<br>".join(lines)


df["reason_display"] = (
    df["reason"].apply(format_reason)
)


# ------------------------------------------------------------
# Boolean columns
# ------------------------------------------------------------

def parse_bool(value):

    if pd.isna(value):

        return False

    if isinstance(value, (bool, np.bool_)):

        return bool(value)

    value = str(value).strip().lower()

    return value in [
        "true",
        "1",
        "yes"
    ]


boolean_cols = [
    "new_predictive_alert",
    "reactive_critical",
    "actionable_before_breach"
]

for col in boolean_cols:

    if col not in df.columns:

        df[col] = False

    df[col] = (
        df[col]
        .apply(parse_bool)
        .astype(bool)
    )


# ------------------------------------------------------------
# Sort
# ------------------------------------------------------------

df = (
    df.dropna(subset=["decision_time"])
    .sort_values(
        ["entity_id", "decision_time"]
    )
    .reset_index(drop=True)
)

if df.empty:

    raise ValueError(
        "No valid decision_time records."
    )


# ============================================================
# 4. PREDICTED BREACH TIME
# ============================================================

df["minutes_to_breach_raw"] = (
    df["minutes_to_predicted_breach"].copy()
)

df["minutes_to_predicted_breach"] = (
    df["minutes_to_predicted_breach"]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)

df["predicted_breach_time"] = (
    df["decision_time"]
    +
    pd.to_timedelta(
        df["minutes_to_predicted_breach"],
        unit="m"
    )
)


# ============================================================
# 5. NOTIFICATION CONDITION
# ============================================================

# Alert candidates

df["is_alert_candidate"] = (
    (df["alert_state"] == "CRITICAL")
    |
    (
        (df["alert_state"] == "WARNING")
        &
        (
            df["quantity"]
            > NOTIFICATION_MIN_QUANTITY
        )
    )
)


# Actual notifications after server logic/cooldown

df["is_notification"] = (
    df["is_alert_candidate"]
    &
    (
        df["new_predictive_alert"]
        |
        df["reactive_critical"]
    )
)


# ============================================================
# 6. RECOVERY CONDITION
# ============================================================

# EXACTLY:
#
# quantity == 0
# AND reason != "No action: covered by pending order"
#
# NO alert_state == NORMAL requirement.
# NO time-gap validation.

df["is_recovery"] = (
    (
        df["reason"]
        != "No action: covered by pending order"
    )
    &
    (
        df["quantity"] == 0
    )
).fillna(False).astype(bool)


# ============================================================
# 7. ALERT EPISODE / RESOLUTION DETECTION
# ============================================================

df["episode_id"] = pd.NA

df["lifecycle"] = "NORMAL"

df["recovery_streak"] = 0

df["alert_resolved"] = False

episodes = []

notification_links = []

episode_counter = 0


for entity_id, group in df.groupby(
    "entity_id",
    sort=False
):

    active = False

    streak = 0

    episode_id = None

    first_notification_time = None

    last_notification_time = None

    notification_count = 0

    episode_notifications = []


    for idx, row in group.iterrows():

        now = row["decision_time"]

        is_notification = bool(
            row["is_notification"]
        )

        is_recovery = bool(
            row["is_recovery"]
        )


        # ----------------------------------------------------
        # A. NEW NOTIFICATION
        # ----------------------------------------------------

        if is_notification:

            if not active:

                episode_counter += 1

                episode_id = (
                    f"{entity_id}:{episode_counter}"
                )

                active = True

                first_notification_time = now

                notification_count = 0

                episode_notifications = []


            notification_count += 1

            last_notification_time = now


            if row["reactive_critical"]:

                notification_type = "Reactive Critical"

            else:

                notification_type = "Predictive Alert"


            episode_notifications.append({
                "notification_time": now,
                "notification_type": notification_type,
                "episode_id": episode_id,
                "entity_id": entity_id,
                "notification_index": notification_count
            })


            # Notification resets recovery streak

            streak = 0


        # ----------------------------------------------------
        # B. NO ACTIVE EPISODE
        # ----------------------------------------------------

        if not active:

            continue


        df.at[idx, "episode_id"] = episode_id


        # ----------------------------------------------------
        # C. RECOVERY STREAK
        # ----------------------------------------------------

        if is_recovery and not is_notification:

            streak += 1

        else:

            streak = 0


        df.at[idx, "recovery_streak"] = streak


        # ----------------------------------------------------
        # D. RESOLVED
        # ----------------------------------------------------

        if streak >= RECOVERY_REQUIRED:

            df.at[idx, "alert_resolved"] = True

            df.at[idx, "lifecycle"] = "RESOLVED"


            resolution_minutes = (
                now - first_notification_time
            ).total_seconds() / 60


            minutes_since_last_notification = (
                now - last_notification_time
            ).total_seconds() / 60


            episodes.append({
                "entity_id": entity_id,
                "episode_id": episode_id,
                "notification_time":
                    first_notification_time,
                "last_notification_time":
                    last_notification_time,
                "resolved_time": now,
                "notification_count":
                    notification_count,
                "resolution_minutes":
                    resolution_minutes,
                "minutes_since_last_notification":
                    minutes_since_last_notification,
                "resolved_count":
                    row["Count"],
                "resolved_reason":
                    row["reason"]
            })


            # Connect all notifications in episode
            # to the resolution event.

            for notification in episode_notifications:

                notification_links.append({
                    **notification,
                    "resolved_time": now,
                    "resolution_minutes": (
                        now
                        - notification["notification_time"]
                    ).total_seconds() / 60
                })


            # Close episode

            active = False

            streak = 0

            episode_id = None

            episode_notifications = []


        elif streak > 0:

            df.at[idx, "lifecycle"] = "RECOVERING"

        else:

            df.at[idx, "lifecycle"] = "ACTIVE"


episodes_df = pd.DataFrame(episodes)

notification_links_df = pd.DataFrame(
    notification_links
)


# ============================================================
# 8. FILTERED DATA / EVENTS
# ============================================================

filtered_df = df[
    df["is_alert_candidate"]
].copy()


warning_points = filtered_df[
    filtered_df["alert_state"] == "WARNING"
].copy()


critical_points = filtered_df[
    filtered_df["alert_state"] == "CRITICAL"
].copy()


notification_points = df[
    df["is_notification"]
].copy()


predictive_points = notification_points[
    notification_points["new_predictive_alert"]
].copy()


reactive_points = notification_points[
    notification_points["reactive_critical"]
].copy()


replenishment_points = filtered_df[
    filtered_df["quantity"] > 0
].copy()


completion_points = filtered_df[
    filtered_df["expected_completion_time"].notna()
].copy()


breach_points = filtered_df[
    filtered_df["predicted_breach_time"].notna()
].copy()


order_completion_points = replenishment_points[
    replenishment_points[
        "expected_completion_time"
    ].notna()
].copy()


resolved_points = df[
    df["alert_resolved"]
].copy()


# ============================================================
# 9. SUMMARY
# ============================================================

print("=" * 75)

print(
    "CART COUNTING — NOTIFICATION / RESOLUTION SUMMARY"
)

print("=" * 75)

print(f"Total records: {len(df)}")

print(
    f"Alert candidates: "
    f"{len(filtered_df)}"
)

print(
    f"Actual notifications: "
    f"{len(notification_points)}"
)

print(
    f"Predictive notifications: "
    f"{len(predictive_points)}"
)

print(
    f"Reactive critical notifications: "
    f"{len(reactive_points)}"
)

print(
    f"Resolved episodes: "
    f"{len(episodes_df)}"
)


if not episodes_df.empty:

    print(
        "\nRESOLUTION DURATION "
        "FROM FIRST NOTIFICATION"
    )

    print(
        episodes_df[
            "resolution_minutes"
        ].describe(
            percentiles=[
                0.25,
                0.5,
                0.75,
                0.9
            ]
        )
    )


    print(
        "\nRESOLUTION DURATION "
        "FROM LAST NOTIFICATION"
    )

    print(
        episodes_df[
            "minutes_since_last_notification"
        ].describe(
            percentiles=[
                0.25,
                0.5,
                0.75,
                0.9
            ]
        )
    )


    print("\nEPISODE DETAILS")

    print(
        episodes_df[
            [
                "episode_id",
                "notification_time",
                "last_notification_time",
                "resolved_time",
                "notification_count",
                "resolution_minutes"
            ]
        ].to_string(index=False)
    )


# ============================================================
# 10. HOVER DATA
# ============================================================

def create_hover_data(data):

    result = data.copy()


    result["expected_completion_str"] = (
        result["expected_completion_time"]
        .dt.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
        .fillna("N/A")
    )


    result["predicted_breach_str"] = (
        result["predicted_breach_time"]
        .dt.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
        .fillna("N/A")
    )


    def format_breach_minutes(value):

        if pd.isna(value):

            return "N/A"

        if np.isposinf(value):

            return "No breach predicted"

        if np.isneginf(value):

            return "N/A"

        return f"{value:.1f}"


    result["minutes_to_breach_display"] = (
        result["minutes_to_breach_raw"]
        .apply(format_breach_minutes)
    )


    result["risk_display"] = (
        result["shortage_risk"]
        .apply(
            lambda x: (
                f"{x:.1%}"
                if pd.notna(x)
                and np.isfinite(x)
                else "N/A"
            )
        )
    )


    result["episode_display"] = (
        result["episode_id"]
        .astype("string")
        .fillna("N/A")
    )


    return result[
        [
            "Count",
            "alert_state",
            "risk_display",
            "quantity",
            "urgency",
            "new_predictive_alert",
            "reactive_critical",
            "actionable_before_breach",
            "minutes_to_breach_display",
            "expected_completion_str",
            "predicted_breach_str",
            "reason_display",
            "episode_display",
            "recovery_streak",
            "lifecycle"
        ]
    ].astype(str).to_numpy()


HOVER_TEMPLATE = (
    "<b>Time:</b> "
    "%{x|%Y-%m-%d %H:%M:%S}<br>"

    "<b>Cart Count:</b> "
    "%{customdata[0]}<br>"

    "<b>Alert:</b> "
    "%{customdata[1]}<br>"

    "<b>Shortage Risk:</b> "
    "%{customdata[2]}<br>"

    "<b>Quantity:</b> "
    "%{customdata[3]}<br>"

    "<b>Urgency:</b> "
    "%{customdata[4]}<br>"

    "<b>New Predictive:</b> "
    "%{customdata[5]}<br>"

    "<b>Reactive Critical:</b> "
    "%{customdata[6]}<br>"

    "<b>Actionable:</b> "
    "%{customdata[7]}<br>"

    "<b>Minutes to Breach:</b> "
    "%{customdata[8]}<br>"

    "<b>Expected Completion:</b> "
    "%{customdata[9]}<br>"

    "<b>Predicted Breach:</b> "
    "%{customdata[10]}<br>"

    "<b>Episode:</b> "
    "%{customdata[12]}<br>"

    "<b>Recovery Streak:</b> "
    "%{customdata[13]}<br>"

    "<b>Lifecycle:</b> "
    "%{customdata[14]}<br>"

    "<br><b>Reason:</b><br>"
    "%{customdata[11]}"

    "<extra></extra>"
)


# ============================================================
# 11. CREATE DASHBOARD — 2 CHARTS
# ============================================================

fig = make_subplots(
    rows=2,
    cols=1,

    shared_xaxes=True,

    vertical_spacing=0.12,

    row_heights=[
        0.60,
        0.40
    ],

    subplot_titles=[
        "1. Cart Availability & Alert Resolution",
        "2. Notification → Resolved Timeline"
    ]
)


# ============================================================
# 12. CHART 1 — ACTUAL CART COUNT
# ============================================================

# Separate entities to avoid connecting
# unrelated cart-count series.

for entity_id, group in df.groupby(
    "entity_id",
    sort=False
):

    fig.add_trace(
        go.Scatter(
            x=group["decision_time"],

            y=group["Count"],

            mode="lines",

            name=(
                f"Actual Cart Count ({entity_id})"
            ),

            line=dict(
                color="#64748B",
                width=2.5
            ),

            hovertemplate=(
                "<b>Time:</b> "
                "%{x|%Y-%m-%d %H:%M:%S}<br>"

                "<b>Cart Count:</b> "
                "%{y:.0f}<br>"

                f"<b>Entity:</b> "
                f"{html.escape(entity_id)}"

                "<extra></extra>"
            )
        ),

        row=1,
        col=1
    )


# ============================================================
# 13. CHART 1 — WARNING
# ============================================================

if not warning_points.empty:

    fig.add_trace(
        go.Scatter(
            x=warning_points["decision_time"],

            y=warning_points["Count"],

            mode="markers",

            name=(
                f"WARNING ({len(warning_points)})"
            ),

            marker=dict(
                color="#FACC15",
                size=12,
                symbol="circle",

                line=dict(
                    color="#854D0E",
                    width=1.5
                )
            ),

            customdata=create_hover_data(
                warning_points
            ),

            hovertemplate=HOVER_TEMPLATE
        ),

        row=1,
        col=1
    )


# ============================================================
# 14. CHART 1 — CRITICAL
# ============================================================

if not critical_points.empty:

    fig.add_trace(
        go.Scatter(
            x=critical_points["decision_time"],

            y=critical_points["Count"],

            mode="markers",

            name=(
                f"CRITICAL ({len(critical_points)})"
            ),

            marker=dict(
                color="#DC2626",
                size=14,
                symbol="circle",

                line=dict(
                    color="#7F1D1D",
                    width=1.5
                )
            ),

            customdata=create_hover_data(
                critical_points
            ),

            hovertemplate=HOVER_TEMPLATE
        ),

        row=1,
        col=1
    )


# ============================================================
# 15. CHART 1 — RESOLVED
# ============================================================

if not resolved_points.empty:

    fig.add_trace(
        go.Scatter(
            x=resolved_points["decision_time"],

            y=resolved_points["Count"],

            mode="markers",

            name=(
                f"RESOLVED ({len(resolved_points)})"
            ),

            marker=dict(
                color="#16A34A",
                size=17,
                symbol="circle",

                line=dict(
                    color="#14532D",
                    width=2
                )
            ),

            customdata=create_hover_data(
                resolved_points
            ),

            hovertemplate=(
                "<b>RESOLVED</b><br>"
                + HOVER_TEMPLATE
            )
        ),

        row=1,
        col=1
    )


# ============================================================
# 16. CHART 1 — THRESHOLDS
# ============================================================

fig.add_hline(
    y=AMBER_THRESHOLD,

    line_dash="dash",
    line_color="#F59E0B",
    line_width=1.5,

    annotation_text=(
        f"AMBER = {AMBER_THRESHOLD}"
    ),

    annotation_position="top left",

    row=1,
    col=1
)


fig.add_hline(
    y=RED_THRESHOLD,

    line_dash="dash",
    line_color="#DC2626",
    line_width=1.5,

    annotation_text=(
        f"RED = {RED_THRESHOLD}"
    ),

    annotation_position="bottom left",

    row=1,
    col=1
)


# ============================================================
# 17. CHART 2 — EVENT TIMELINE
# ============================================================

EVENT_Y = {
    "Predictive Alert": 6,
    "Reactive Critical": 5,
    "Replenishment Order": 4,
    "Expected Completion": 3,
    "Predicted Breach": 2,
    "Resolved Alert": 1
}


def add_timeline_events(
    points,
    event_name,
    time_column="decision_time",
    marker_colors=None
):

    if points.empty:

        return


    points = points[
        points[time_column].notna()
    ].copy()


    if points.empty:

        return


    style = EVENT_STYLES[event_name]


    if marker_colors is None:

        colors = style["color"]

    else:

        colors = (
            pd.Series(marker_colors)
            .reindex(points.index)
            .fillna(style["color"])
            .tolist()
        )


    fig.add_trace(
        go.Scatter(
            x=points[time_column],

            y=[
                EVENT_Y[event_name]
            ] * len(points),

            mode="markers",

            name=(
                f"{event_name} ({len(points)})"
            ),

            marker=dict(
                color=colors,
                size=style["size"],
                symbol=style["symbol"],

                line=dict(
                    color="white",
                    width=1.5
                )
            ),

            customdata=create_hover_data(
                points
            ),

            hovertemplate=(
                f"<b>{event_name}</b><br>"
                + HOVER_TEMPLATE
            )
        ),

        row=2,
        col=1
    )


# ============================================================
# 18. CHART 2 — EVENTS
# ============================================================

# Predictive notifications

add_timeline_events(
    predictive_points,
    "Predictive Alert"
)


# Reactive critical notifications

add_timeline_events(
    reactive_points,
    "Reactive Critical"
)


# Replenishment orders

order_colors = (
    replenishment_points["urgency"]
    .map(URGENCY_COLORS)
    .fillna("#94A3B8")
)

add_timeline_events(
    replenishment_points,
    "Replenishment Order",
    marker_colors=order_colors
)


# Expected completion

add_timeline_events(
    completion_points,
    "Expected Completion",
    time_column="expected_completion_time"
)


# Predicted breach

add_timeline_events(
    breach_points,
    "Predicted Breach",
    time_column="predicted_breach_time"
)


# Resolved

add_timeline_events(
    resolved_points,
    "Resolved Alert"
)


# ============================================================
# 19. ORDER → EXPECTED COMPLETION
# ============================================================

# Use one trace for all order-completion links.
# This keeps Plotly responsive.

connection_x = []

connection_y = []


for _, row_data in order_completion_points.iterrows():

    start = row_data["decision_time"]

    end = row_data["expected_completion_time"]


    if (
        pd.isna(start)
        or pd.isna(end)
        or end < start
    ):

        continue


    connection_x.extend([
        start,
        end,
        None
    ])


    connection_y.extend([
        EVENT_Y["Replenishment Order"],
        EVENT_Y["Expected Completion"],
        None
    ])


if connection_x:

    fig.add_trace(
        go.Scatter(
            x=connection_x,

            y=connection_y,

            mode="lines",

            name="Order → Completion",

            line=dict(
                color="#94A3B8",
                width=1.5,
                dash="dot"
            ),

            hoverinfo="skip"
        ),

        row=2,
        col=1
    )


# ============================================================
# 20. NOTIFICATION → RESOLVED
# ============================================================

# Connect every notification within an episode
# to the episode's resolution time.

resolved_connection_x = []

resolved_connection_y = []


if not notification_links_df.empty:

    for _, link in notification_links_df.iterrows():

        if (
            link["notification_type"]
            == "Reactive Critical"
        ):

            start_y = EVENT_Y[
                "Reactive Critical"
            ]

        else:

            start_y = EVENT_Y[
                "Predictive Alert"
            ]


        resolved_connection_x.extend([
            link["notification_time"],
            link["resolved_time"],
            None
        ])


        resolved_connection_y.extend([
            start_y,
            EVENT_Y["Resolved Alert"],
            None
        ])


if resolved_connection_x:

    fig.add_trace(
        go.Scatter(
            x=resolved_connection_x,

            y=resolved_connection_y,

            mode="lines",

            name="Notification → Resolved",

            line=dict(
                color="#16A34A",
                width=1.5,
                dash="dash"
            ),

            hoverinfo="skip"
        ),

        row=2,
        col=1
    )


# ============================================================
# 21. CHART 2 — EVENT AXIS
# ============================================================

fig.update_yaxes(
    tickmode="array",

    tickvals=[
        1, 2, 3, 4, 5, 6
    ],

    ticktext=[
        "Resolved Alert",
        "Predicted Breach",
        "Expected Completion",
        "Replenishment Order",
        "Reactive Critical",
        "Predictive Alert"
    ],

    range=[
        0.5,
        6.5
    ],

    showgrid=True,
    gridcolor="#F1F5F9",

    row=2,
    col=1
)


# ============================================================
# 22. DASHBOARD LAYOUT
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "Cart Counting — Alerts & Resolution Timeline"
            "<br>"
            f"<sup>Resolved after "
            f"{RECOVERY_REQUIRED} consecutive recovery records"
            " (no time-gap validation)</sup>"
        ),

        font=dict(
            size=22
        )
    ),

    template="plotly_white",

    height=1150,

    hovermode="closest",

    dragmode="zoom",

    legend=dict(
        orientation="h",

        yanchor="bottom",
        y=1.02,

        xanchor="left",
        x=0
    ),

    margin=dict(
        l=190,
        r=70,
        t=180,
        b=110
    ),

    uirevision="cart_dashboard"
)


# ============================================================
# 23. X AXES — ZOOM / PAN
# ============================================================

fig.update_xaxes(
    type="date",

    showgrid=True,
    gridcolor="#E5E7EB",

    tickformat="%m-%d %H:%M",

    hoverformat="%Y-%m-%d %H:%M:%S",

    showticklabels=True,

    ticks="outside",

    tickangle=-30,

    automargin=True,

    autorange=True,

    fixedrange=False
)


for row in range(1, 3):

    fig.update_xaxes(
        showticklabels=True,
        fixedrange=False,

        row=row,
        col=1
    )


# ============================================================
# 24. RANGE SLIDER
# ============================================================

fig.update_xaxes(
    rangeslider=dict(
        visible=True,
        thickness=0.08
    ),

    title_text="Decision / Event Time",

    row=2,
    col=1
)


# ============================================================
# 25. CART COUNT Y AXIS
# ============================================================

fig.update_yaxes(
    title_text="Available Cart Count",

    rangemode="tozero",

    row=1,
    col=1
)


# ============================================================
# 26. EXPORT
# ============================================================

fig.write_html(
    OUTPUT_FILE,

    include_plotlyjs=True,

    full_html=True,

    config=PLOTLY_CONFIG
)


print(
    f"\nDashboard saved: {OUTPUT_FILE}"
)


fig.show(
    config=PLOTLY_CONFIG
)

CART COUNTING — NOTIFICATION / RESOLUTION SUMMARY
Total records: 8708
Alert candidates: 356
Actual notifications: 204
Predictive notifications: 127
Reactive critical notifications: 77
Resolved episodes: 109

RESOLUTION DURATION FROM FIRST NOTIFICATION
count    109.000000
mean      41.691131
std       29.743100
min       13.350000
25%       20.016667
50%       30.033333
75%       53.400000
90%       88.086667
max      150.150000
Name: resolution_minutes, dtype: float64

RESOLUTION DURATION FROM LAST NOTIFICATION
count    109.000000
mean      26.876147
std       17.118321
min       13.350000
25%       16.700000
50%       20.016667
75%       26.683333
90%       43.383333
max      106.766667
Name: minutes_since_last_notification, dtype: float64

EPISODE DETAILS
        episode_id   notification_time last_notification_time       resolved_time  notification_count  resolution_minutes
  default_entity:1 2026-09-29 16:18:01    2026-09-29 16:18:01 2026-09-29 16:54:43                   1         